In [4]:
import asyncio
import time

MAX_CONCURRENT_LLM_CALLS = 5


async def mock_llm_merge(a, b):
    """Simulates an LLM merge call."""
    await asyncio.sleep(1)  # simulate latency
    return f"merge({a},{b})"


async def merge_pair(a, b, semaphore):
    """Merge two timelines while respecting concurrency limit."""
    async with semaphore:
        result = await mock_llm_merge(a, b)
        return result


async def merge_level(nodes, semaphore):
    """Merge one level of the binary tree."""
    
    tasks = []
    next_level = []

    i = 0
    while i < len(nodes):
        
        # Handle odd node
        if i + 1 >= len(nodes):
            next_level.append(nodes[i])
            break
        
        a = nodes[i]
        b = nodes[i + 1]

        tasks.append(merge_pair(a, b, semaphore))

        i += 2

    results = await asyncio.gather(*tasks)
    next_level.extend(results)

    return next_level


async def binary_merge_all(nodes):
    """Perform full binary tree merging."""

    semaphore = asyncio.Semaphore(MAX_CONCURRENT_LLM_CALLS)

    level = 1

    while len(nodes) > 1:
        start_time = time.time()
        print(f"\nLevel {level} | Nodes: {nodes}")
        nodes = await merge_level(nodes, semaphore)
        end_time = time.time()
        print(f"Time taken for level {level}: {end_time - start_time} seconds\n")
        level += 1

    return nodes[0]


async def main():

    # Example: 8 chunk timelines
    chunks = [f"chunk_{i}" for i in range(1,65)]

    final_result = await binary_merge_all(chunks)

    print("\nFinal Timeline:", final_result)


await main()


Level 1 | Nodes: ['chunk_1', 'chunk_2', 'chunk_3', 'chunk_4', 'chunk_5', 'chunk_6', 'chunk_7', 'chunk_8', 'chunk_9', 'chunk_10', 'chunk_11', 'chunk_12', 'chunk_13', 'chunk_14', 'chunk_15', 'chunk_16', 'chunk_17', 'chunk_18', 'chunk_19', 'chunk_20', 'chunk_21', 'chunk_22', 'chunk_23', 'chunk_24', 'chunk_25', 'chunk_26', 'chunk_27', 'chunk_28', 'chunk_29', 'chunk_30', 'chunk_31', 'chunk_32', 'chunk_33', 'chunk_34', 'chunk_35', 'chunk_36', 'chunk_37', 'chunk_38', 'chunk_39', 'chunk_40', 'chunk_41', 'chunk_42', 'chunk_43', 'chunk_44', 'chunk_45', 'chunk_46', 'chunk_47', 'chunk_48', 'chunk_49', 'chunk_50', 'chunk_51', 'chunk_52', 'chunk_53', 'chunk_54', 'chunk_55', 'chunk_56', 'chunk_57', 'chunk_58', 'chunk_59', 'chunk_60', 'chunk_61', 'chunk_62', 'chunk_63', 'chunk_64']
Time taken for level 1: 7.009878635406494 seconds


Level 2 | Nodes: ['merge(chunk_1,chunk_2)', 'merge(chunk_3,chunk_4)', 'merge(chunk_5,chunk_6)', 'merge(chunk_7,chunk_8)', 'merge(chunk_9,chunk_10)', 'merge(chunk_11,chunk

## Stage 1 — Flatten (Python)

Convert:

chunk → date → events

into:

(date, tag, summary)

Example:

2022-03-04 | imaging | MRI knee
2022-03-04 | imaging | MRI of left knee
2022-03-04 | admission | patient admitted

In [2]:
import json
from pathlib import Path

def flatten_timeline(input_path: str):
    """
    Input shape: [chunk_obj, chunk_obj, ...]
    chunk_obj -> events_by_date[] -> events[]
    Output: flat list of (date, time, tag, summary)
    """
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    flat_rows = []

    for chunk_obj in data:
        for d in chunk_obj.get("events_by_date", []):
            date = d.get("date")
            for ev in d.get("events", []):
                flat_rows.append({
                    "date": date,
                    "time": ev.get("event_time"),      # can be None
                    "tag": ev.get("event_tag"),
                    "summary": ev.get("event_summary"),
                })

    return flat_rows


# --- run ---
input_file = "/home/ankit/smartsense_code/fraudx_timeline_poc/rough_jsons/13871/ld_clean_extract_dates_results_top15.json"
rows = flatten_timeline(input_file)

# Optional: print like your example
for r in rows:
    time_value = "null" if r["time"] is None else r["time"]
    print(f"{r['date']} | {time_value} | {r['tag']} | {r['summary']}")

print(f"\nTotal flattened events: {len(rows)}")

2021-10-29 | null | admission | Patient Alex Tipantaxi admitted to the Emergency Department at St Barnabas Hospital under the care of provider Paul Beyer.
2021-10-29 | null | injury | Right arm pain following a beam accident reported at St. Barnabas Hospital emergency department.
2021-10-29 | null | admission | Emergency department admission of Alex Tipantaxi at St. Barnabas Hospital.
2021-10-29 | 13:17 | injury | A metal beam fell on top of the patient's right arm at work, causing right arm pain.
2021-10-29 | 13:17 | admission | Patient arrived at the Emergency Department via bus at 13:17 for evaluation of right arm injury caused by metal beam accident.
2021-10-29 | 13:37 | admission | Patient TIPANTAXI, ALEX admitted to the Emergency Department at St Barnabas Hospital under provider Beyer, Paul.
2021-10-29 | 13:43 | injury | Patient sustained right arm injury after a metal beam fell on the arm at work; patient able to move the affected extremity with no deformity observed.
2021-10-29